# 00 — Context

## Revision Demystifies Small Language Models

This notebook initializes the `pruning-rml` repo around arXiv:2606.14150,  
*Small LLMs: Pruning vs. Training from Scratch*.

It does **not** try to reproduce large GPU pruning experiments yet.

It establishes the repo vocabulary:

- **reset**: train a small model from random initialization
- **revision**: prune a larger pretrained model, retrain, and evaluate
- **residue**: capability or structure that persists after compression

In [ ]:
# Cell 1 — setup

from pathlib import Path
import sys
import os
import subprocess

REPO_URL = "https://github.com/thinkthoughts/pruning-rml.git"
REPO_NAME = "pruning-rml"

def find_repo_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for path in [start, *start.parents]:
        if (path / "src").exists() and (
            (path / "paper.yaml").exists() or (path / ".git").exists()
        ):
            return path
    return start

cwd = Path.cwd().resolve()

# If this is a fresh Colab runtime and the repo is not present, clone it.
# This is safe to skip when running locally inside the repo.
if not (cwd / "src").exists() and not (cwd / REPO_NAME / "src").exists():
    try:
        subprocess.run(["git", "clone", REPO_URL], check=True)
    except Exception as exc:
        print("Clone skipped or failed:", exc)

# If clone created pruning-rml/ under cwd, move into it.
if (cwd / REPO_NAME / "src").exists():
    os.chdir(cwd / REPO_NAME)

repo_root = find_repo_root()
os.chdir(repo_root)

src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print("cwd       :", Path.cwd())
print("repo_root :", repo_root)
print("src_path  :", src_path)
print("src exists:", src_path.exists())
print("files     :", [p.name for p in repo_root.iterdir()][:20])

In [ ]:
# Cell 2 — fallback src creation

# This cell makes the notebook runnable even if src/pruning_rml has not
# been committed yet. If the files already exist, it leaves them alone.

pkg = repo_root / "src" / "pruning_rml"
pkg.mkdir(parents=True, exist_ok=True)

files = {
    "__init__.py": '''"""pruning_rml: utilities for studying pruning as revision."""\n\n__version__ = "0.1.0"\n''',
    "paths.py": '''"""Path helpers for notebooks and scripts."""\n\nfrom pathlib import Path\n\ndef ensure_dirs(root, names=("results", "figures", "reports")):\n    root = Path(root)\n    out = {}\n    for name in names:\n        path = root / name\n        path.mkdir(parents=True, exist_ok=True)\n        out[name] = path\n    return out\n''',
    "paper.py": '''"""Load paper and statement metadata."""\n\nfrom pathlib import Path\n\ndef load_yaml(path):\n    path = Path(path)\n    if not path.exists():\n        raise FileNotFoundError(f"Missing YAML file: {path}")\n    try:\n        import yaml\n    except ImportError as exc:\n        raise ImportError("Install pyyaml to load paper.yaml: pip install pyyaml") from exc\n    with path.open("r", encoding="utf-8") as f:\n        data = yaml.safe_load(f)\n    return data or {}\n\ndef paper_summary_table(paper):\n    return {\n        "identifier": paper.get("identifier", "—"),\n        "title": paper.get("paper_title", paper.get("title", "—")),\n        "category": paper.get("category", "—"),\n        "status": paper.get("status", "—"),\n    }\n''',
    "comparisons.py": '''"""Reset versus revision comparisons."""\n\nfrom dataclasses import dataclass\n\n@dataclass(frozen=True)\nclass Comparison:\n    method: str\n    score: float\n    tokens: int\n    mode: str = "unspecified"\n\ndef winner(a, b):\n    return a if a.score >= b.score else b\n\ndef label_mode(method):\n    lower = method.lower()\n    if "scratch" in lower or "random" in lower or "reset" in lower:\n        return "reset"\n    if "prun" in lower or "revision" in lower or "inherited" in lower:\n        return "revision"\n    return "observe"\n''',
    "visualization.py": '''"""Lightweight plotting helpers."""\n\ndef save_bar_chart(df, x, y, path, title=None, xlabel=None, ylabel=None):\n    import matplotlib.pyplot as plt\n    fig, ax = plt.subplots(figsize=(8, 5))\n    ax.bar(df[x], df[y])\n    if title:\n        ax.set_title(title)\n    ax.set_xlabel(xlabel or x)\n    ax.set_ylabel(ylabel or y)\n    ax.tick_params(axis="x", rotation=20)\n    fig.tight_layout()\n    fig.savefig(path, dpi=160)\n    plt.close(fig)\n    return path\n''',
}

for name, content in files.items():
    path = pkg / name
    if not path.exists():
        path.write_text(content, encoding="utf-8")

# Also create paper.yaml if missing, so the notebook runs standalone.
paper_yaml = repo_root / "paper.yaml"
if not paper_yaml.exists():
    paper_yaml.write_text('''id: 2606-14150
date: "2026-06-16"
source: "arXiv"
identifier: "2606.14150"
title: "Revision Demystifies Small Language Models"
paper_title: "Small LLMs: Pruning vs. Training from Scratch"
category: "cs.LG"
status: "draft"
links:
  paper_abs: "https://arxiv.org/abs/2606.14150"
  paper_html: "https://arxiv.org/html/2606.14150v1"
  repo: "https://github.com/zlab-princeton/llm-pruning-collection"
claim: >
  Smaller language models can be built through reset or revision. When
  training-token budgets are matched, revision through pruning preserves useful
  structure that random initialization must rediscover through additional training.
''', encoding="utf-8")

# Refresh import path after creating files.
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print("pruning_rml package ready:", pkg.exists())

In [ ]:
# Cell 3 — imports

import pandas as pd
import matplotlib.pyplot as plt

from pruning_rml.paths import ensure_dirs
from pruning_rml.paper import load_yaml, paper_summary_table
from pruning_rml.comparisons import label_mode
from pruning_rml.visualization import save_bar_chart

outputs = ensure_dirs(repo_root)
print(outputs)

In [ ]:
# Cell 4 — load paper metadata

paper = load_yaml(repo_root / "paper.yaml")
pd.DataFrame([paper_summary_table(paper)])

## Reset vs revision

The paper's central comparison becomes simple in repo vocabulary:

| Mode | Construction | Question |
|---|---|---|
| reset | train from scratch | Can capability be rebuilt? |
| revision | prune larger pretrained model | Which capabilities survive compression? |
| observe | evaluate under budgets | When does revision outperform reset? |

In [ ]:
# Cell 5 — reset / observe / revision table

comparison_rows = [
    {
        "section": "Reset",
        "method": "training from scratch",
        "mode": label_mode("training from scratch"),
        "construction": "random initialization + training tokens",
        "question": "Can capability be rebuilt?",
    },
    {
        "section": "Observe",
        "method": "matched-budget evaluation",
        "mode": label_mode("matched-budget evaluation"),
        "construction": "same-budget comparison",
        "question": "When does revision outperform reset?",
    },
    {
        "section": "Revision",
        "method": "pruning larger pretrained model",
        "mode": label_mode("pruning larger pretrained model"),
        "construction": "inherited structure + compression + retraining",
        "question": "Which capabilities survive compression?",
    },
]

comparison_df = pd.DataFrame(comparison_rows)
comparison_df

## Toy observation table

This is **not** a reproduction of the paper's benchmark values.

It is a placeholder table that makes the notebook runnable and establishes
the plotting pattern for later notebooks.

In [ ]:
# Cell 6 — toy result table

toy_results = pd.DataFrame([
    {
        "method": "scratch",
        "mode": "reset",
        "relative_score": 0.72,
        "note": "random initialization under matched budget",
    },
    {
        "method": "pruned + retrained",
        "mode": "revision",
        "relative_score": 0.84,
        "note": "inherits structure from larger pretrained model",
    },
    {
        "method": "scratch + larger budget",
        "mode": "reset",
        "relative_score": 0.82,
        "note": "scratch becomes more competitive with more training",
    },
])

toy_results

In [ ]:
# Cell 7 — save toy figure

fig_path = outputs["figures"] / "00_reset_vs_revision_toy.png"

save_bar_chart(
    toy_results,
    x="method",
    y="relative_score",
    path=fig_path,
    title="Toy comparison: reset vs revision",
    xlabel="Method",
    ylabel="Relative score",
)

fig_path

In [ ]:
# Cell 8 — display saved figure

from IPython.display import Image, display

display(Image(filename=str(fig_path)))

In [ ]:
# Cell 9 — write summary markdown

summary = f'''# Notebook 00 — Context

## {paper.get("title", "Revision Demystifies Small Language Models")}

This notebook establishes the first working loop for `pruning-rml`.

## Core vocabulary

- **Reset**: train a smaller model from scratch.
- **Revision**: prune a larger pretrained model, retrain, and evaluate.
- **Residue**: capability or structure that survives compression.

## Core claim

{paper.get("claim", "Pruning can preserve inherited structure under matched budgets.")}

## Next notebook

`07_pruning_vs_scratch.ipynb`
'''

summary_path = outputs["reports"] / "00_context.md"
summary_path.write_text(summary, encoding="utf-8")

summary_path

## Result

Notebook 00 establishes:

```text
paper.yaml
    ↓
src helpers
    ↓
context table
    ↓
toy comparison figure
    ↓
summary markdown
```

Next notebook:

`07_pruning_vs_scratch.ipynb`